In [3]:
!pip install gradio faiss-cpu openai langchain

import csv
import os
import faiss
import pickle
from openai import OpenAI
from langchain.prompts import PromptTemplate

# Initialize OpenAI client
client = OpenAI(api_key='Api_key')  # Use environment variable in production!

# Files
FEEDBACK_LOG = "meme_feedback.csv"
INDEX_FILE = "faiss_index.bin"
CAPTIONS_FILE = "captions.pkl"

# Prompt Template
meme_prompt_template = PromptTemplate(
    input_variables=["caption"],
    template="A funny meme image based on the caption: '{caption}'"
)

# FAISS Index setup
dimension = 1536  # OpenAI text-embedding-3-small returns 1536-dim vectors
if os.path.exists(INDEX_FILE) and os.path.exists(CAPTIONS_FILE):
    index = faiss.read_index(INDEX_FILE)
    with open(CAPTIONS_FILE, "rb") as f:
        all_captions = pickle.load(f)
else:
    index = faiss.IndexFlatL2(dimension)
    all_captions = []

def embed_caption(caption):
    response = client.embeddings.create(
        input=[caption],
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

def store_caption_vector(caption, vector):
    all_captions.append(caption)
    index.add([vector])
    # Save index and captions
    faiss.write_index(index, INDEX_FILE)
    with open(CAPTIONS_FILE, "wb") as f:
        pickle.dump(all_captions, f)

def find_similar_captions(caption, threshold=0.85):
    if len(all_captions) == 0:
        return []
    query_vec = embed_caption(caption)
    D, I = index.search([query_vec], k=5)  # get 5 nearest neighbors
    similar = []
    for dist, idx in zip(D[0], I[0]):
        similarity = 1 / (1 + dist)  # approximate cosine similarity
        if similarity > threshold:
            similar.append(all_captions[idx])
    return similar

def generate_meme(caption):
    similar = find_similar_captions(caption)
    if similar:
        return f"⚠️ This caption is similar to a previous one: '{similar[0]}'", None
    prompt = meme_prompt_template.format(caption=caption)
    response = client.images.generate(
        model="dall-e-3",
        prompt=prompt,
        size="1024x1024",
        quality="standard",
        n=1
    )
    image_url = response.data[0].url
    # Save caption & embedding
    vector = embed_caption(caption)
    store_caption_vector(caption, vector)
    return "", image_url

def submit_feedback(caption, feedback):
    file_exists = os.path.isfile(FEEDBACK_LOG)
    with open(FEEDBACK_LOG, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(["Caption", "Feedback"])
        writer.writerow([caption, feedback])
    return "✅ Feedback recorded!"

# Gradio Interface
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 MemeBot with FAISS - Meme Generator + Caption Similarity Check")

    caption_input = gr.Textbox(label="Enter your meme caption")
    similarity_output = gr.Textbox(label="Similarity Alert", interactive=False)
    image_output = gr.Image(label="Generated Meme")
    feedback_buttons = gr.Radio(["yes", "no"], label="Did you find it funny?")
    feedback_message = gr.Textbox(visible=False)

    def generate_and_show(caption):
        message, url = generate_meme(caption)
        return message, url

    def record_feedback(caption, feedback):
        return submit_feedback(caption, feedback)

    generate_button = gr.Button("Generate Meme")
    generate_button.click(
        fn=generate_and_show,
        inputs=caption_input,
        outputs=[similarity_output, image_output]
    )

    submit_button = gr.Button("Submit Feedback")
    submit_button.click(
        fn=record_feedback,
        inputs=[caption_input, feedback_buttons],
        outputs=feedback_message
    )

demo.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://00739851ec40863132.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
